# Ames Home Price Estimator

This notebook follows the project from looking at the data to saving a model. Run the cells from top to bottom. The Streamlit app in `app.py` loads the model saved near the end.

The project uses simple scikit-learn tools and one fixed train/test split so the steps are easier to follow.


# 1. Data Understanding and Exploratory Data Analysis

## What we are learning
Before modeling, we need to understand what one row represents, what the columns mean, and where the data may need care. The training file has a known sale price for each home. The competition test file has the same kinds of inputs but deliberately hides its prices; it is not used to score our local model.

The project includes Kaggle's `data_description.txt`, which explains the housing fields and category codes. Keep it open while exploring unfamiliar columns.


### 1. Load familiar tools and the two provided datasets
`pandas` organizes rows and columns. NumPy supports numeric work; Seaborn and Matplotlib make patterns easier to see.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "train.csv").exists():
    ROOT = Path.cwd().parent
TRAIN_PATH = ROOT / "data" / "raw" / "train.csv"
TEST_PATH = ROOT / "data" / "raw" / "test.csv"
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(f"Project folder: {ROOT}")
print(f"Training rows: {len(train):,} | competition test rows: {len(test):,}")


### 2. Check the shape, sample rows, and data types
A row describes one sold home. `SalePrice` is the value we want to predict, and `Id` is a record identifier rather than a property characteristic.


In [ ]:
print("Training shape:", train.shape)
print("Competition test shape:", test.shape)
display(train.head())
display(train.dtypes.value_counts().rename_axis("pandas dtype").to_frame("column count"))
object_features = train.select_dtypes(include="object").columns.tolist()
numeric_features = train.select_dtypes(include="number").columns.tolist()
print("Object/categorical-style columns:", len(object_features), object_features[:12])
print("Numeric columns (including numeric-coded categories):", len(numeric_features), numeric_features[:12])
print("MSSubClass is stored as a number but represents dwelling type; check its codes in the data dictionary.")
print("Target in training:", "SalePrice" in train.columns)
print("Target in competition test:", "SalePrice" in test.columns)


### Read the data dictionary
The dictionary explains individual columns and their category codes. For example, `OverallQual` is a material/finish rating, while `Neighborhood` names a location. Numeric storage does not always mean a quantity: `MSSubClass` is a numeric code for dwelling type.


In [ ]:
dictionary_path = ROOT / "data" / "raw" / "data_description.txt"
dictionary_text = dictionary_path.read_text(encoding="utf-8")
for field in ["MSSubClass:", "OverallQual:", "Neighborhood:"]:
    start = dictionary_text.find(field)
    print(f"\n--- {field} ---")
    print(dictionary_text[start:start + 280] if start >= 0 else "Description not found")


### 3. Missing values and duplicate rows
A blank can mean that information is missing, or that a feature is absent (for example, no pool). The data dictionary helps distinguish these meanings. We will use a simple median or most-common-value rule inside the model pipeline later, so the rule is learned from training rows only.


In [ ]:
missing = train.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].rename("missing training values").to_frame().head(20))
print("Duplicate training rows:", train.duplicated().sum())
print("Duplicate competition test rows:", test.duplicated().sum())


### 4. Descriptive statistics and the target
The mean is sensitive to expensive homes; the median gives a more typical sale price. The distribution is right-skewed, so a few high-priced sales stretch its upper tail.


In [ ]:
display(train.describe().T.head(20))
target_summary = train["SalePrice"].describe()
display(target_summary.to_frame("SalePrice in dollars"))
print(f"Median price: ${train.SalePrice.median():,.0f}; mean price: ${train.SalePrice.mean():,.0f}")
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(train["SalePrice"], bins=35, kde=True, ax=ax, color="#2878B5")
ax.set(title="Sale prices in the training data", xlabel="Sale price (US dollars)")
plt.tight_layout()
plt.show()


### 5. Look at feature distributions
Histograms show the ranges and skew of numeric fields. Boxplots help spot unusual values; an unusual home is not automatically an error, so we should not delete it without a reason.


In [ ]:
numeric_to_plot = ["OverallQual", "GrLivArea", "LotArea", "TotalBsmtSF", "YearBuilt"]
train[numeric_to_plot].hist(figsize=(12, 7), bins=25, color="#66A182", edgecolor="white")
plt.suptitle("Selected numeric property features")
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(8, 3))
sns.boxplot(x=train["SalePrice"], ax=ax, color="#E6A857")
ax.set(title="Sale price spread", xlabel="Sale price (US dollars)")
plt.tight_layout()
plt.show()


### 6. Compare prices across useful features
These plots show association, not proof that a feature causes a higher price. Quality ratings are ordered labels in the data dictionary, while neighborhood is a category.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=train, x="OverallQual", y="SalePrice", ax=axes[0], color="#8AB6D6")
axes[0].set(title="Sale price by overall quality", ylabel="Sale price ($)")
top_neighborhoods = train["Neighborhood"].value_counts().head(8).index
sns.boxplot(data=train[train.Neighborhood.isin(top_neighborhoods)], x="Neighborhood", y="SalePrice", ax=axes[1])
axes[1].tick_params(axis="x", rotation=45)
axes[1].set(title="Prices in the eight most common neighborhoods", ylabel="Sale price ($)")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.regplot(data=train, x="GrLivArea", y="SalePrice", scatter_kws={"alpha": 0.45}, line_kws={"color": "#D14A3B"}, ax=ax)
ax.set(title="Above-ground living area and sale price", xlabel="Living area (sq ft)", ylabel="Sale price ($)")
plt.tight_layout()
plt.show()


### 7. Numeric correlations
Correlation measures straight-line association between numeric values. It does not capture every relationship and does not imply causation. We exclude `Id`, since its numbering does not describe a home.


In [ ]:
correlation = train.select_dtypes(include="number").drop(columns=["Id"]).corr(numeric_only=True)
top_with_target = correlation["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False).head(12)
display(top_with_target.rename("absolute correlation with SalePrice").to_frame())
selected = ["SalePrice"] + top_with_target.index[:9].tolist()
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(train[selected].corr(numeric_only=True), annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Sale price and selected numeric features")
plt.tight_layout()
plt.show()


### What we found and what comes next
The data has 1,460 labeled sales and 1,459 competition examples. There are no duplicate rows. Missingness is concentrated in fields such as pool, alley, fence, and some basement/garage details; some blanks mean that an amenity is absent. Sale prices are right-skewed, and overall quality and living area show clear relationships with price. These patterns motivate careful imputation, category encoding, and a held-out evaluation in the next notebooks.


# 2. Data Cleaning and Preprocessing

## Why preprocessing matters
Most models need numeric inputs and do not accept blank cells. We will define one repeatable pipeline that fills numeric blanks with the median, fills category blanks with the most common value, converts categories into indicator columns, and scales numeric values. The pipeline is fitted only on training rows, which prevents information from the test split leaking into preprocessing.

To keep the Streamlit form understandable, this learning project uses 14 useful fields rather than every available column. `Id` is excluded because it is only a row identifier. The chosen fields include size, age, quality, and location.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "train.csv").exists():
    ROOT = Path.cwd().parent
TRAIN_PATH = ROOT / "data" / "raw" / "train.csv"
TEST_PATH = ROOT / "data" / "raw" / "test.csv"
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(f"Project folder: {ROOT}")
print(f"Training rows: {len(train):,} | competition test rows: {len(test):,}")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Numeric inputs:", NUMERIC_FEATURES)
print("Categorical inputs:", CATEGORICAL_FEATURES)
print("Total selected inputs:", len(FEATURES))


### Separate predictors and target, then make one fixed split
The held-out portion acts like new data: we hide its prices while training and use them only for the final model comparison. A fixed random seed means each notebook recreates the same rows.


In [ ]:
X = train[FEATURES]
y = train[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Training split: {X_train.shape[0]} rows")
print(f"Held-out split: {X_test.shape[0]} rows")


### Inspect missing values in selected inputs
Missing values are real in this dataset. We do not fill them by hand before splitting: the imputer learns the median or most common category from the training portion and applies it to both portions.


In [ ]:
display(X_train.isna().sum().sort_values(ascending=False).to_frame("missing in training portion"))


### Build and inspect the preprocessing pipeline
One-hot encoding creates a 0/1 column for each observed category. `handle_unknown='ignore'` lets the app accept a category not seen during training without crashing. Scaling puts numeric columns on similar ranges, which is useful for linear models. Tree models do not require scaling, but sharing one preprocessing setup keeps the comparison straightforward.


In [ ]:
preprocessor = build_preprocessor()
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
print("Encoded training matrix:", X_train_ready.shape)
print("Encoded held-out matrix:", X_test_ready.shape)
print("Any missing values after preprocessing:", np.isnan(X_train_ready).any())
display(pd.DataFrame(X_train_ready[:3], columns=preprocessor.get_feature_names_out()).iloc[:, :12])


### Outliers and scope
The EDA showed a few very large homes and high sale prices. Those values can represent real properties, so this beginner project keeps them. A more advanced study could verify unusual records against source documentation or compare a target transformation, but we will avoid removing valid examples without evidence.


### Connection to the full workflow
The fitted transformer is not a separate manual step at prediction time. In Notebook 3 it will be bundled with each estimator in a scikit-learn `Pipeline`. This means the training and Streamlit prediction paths apply the same learned medians, category columns, and scaling.


# 3. Model Training and Comparison

## The comparison question
Which of five approachable regression algorithms predicts sale prices best on the same held-out rows? We will use the fixed split from Notebook 2 and place preprocessing inside every model pipeline so each one gets the same data preparation.

Linear Regression fits a weighted straight-line relationship. Ridge and Lasso are linear models that shrink coefficients to reduce overfitting. A Decision Tree makes a sequence of feature-based splits. A Random Forest averages many trees, which can capture more varied patterns but is less easy to explain.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "train.csv").exists():
    ROOT = Path.cwd().parent
TRAIN_PATH = ROOT / "data" / "raw" / "train.csv"
TEST_PATH = ROOT / "data" / "raw" / "test.csv"
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(f"Project folder: {ROOT}")
print(f"Training rows: {len(train):,} | competition test rows: {len(test):,}")


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES = [
    "OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", "YearBuilt",
    "FullBath", "BedroomAbvGr", "LotArea", "Fireplaces",
]
CATEGORICAL_FEATURES = [
    "KitchenQual", "Neighborhood", "ExterQual", "BsmtQual", "CentralAir",
]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "SalePrice"


def build_preprocessor():
    """Fill blanks, scale numeric fields, and convert categories to numbers."""
    numeric = Pipeline([
        ("fill_missing", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("fill_missing", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric, NUMERIC_FEATURES),
        ("categorical", categorical, CATEGORICAL_FEATURES),
    ], remainder="drop", verbose_feature_names_out=False)

print("Numeric fields:", NUMERIC_FEATURES)
print("Category fields:", CATEGORICAL_FEATURES)
print("Total fields:", len(FEATURES))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = train[FEATURES]
y = train[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)


### Create the five model pipelines
`random_state=42` makes tree results repeatable. The tree settings here are simple restraints that reduce extreme memorization; we are demonstrating model comparison, not conducting an extensive tuning search.


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Lasso Regression": Lasso(alpha=100.0, max_iter=20000),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=1),
}

pipelines = {
    name: Pipeline([("preprocess", build_preprocessor()), ("model", estimator)])
    for name, estimator in models.items()
}


### Train, predict, and calculate four metrics
MAE is the average absolute error in dollars and is easy to interpret. MSE squares errors, so large misses count more. RMSE returns that penalty to dollar units. R² compares the model to a simple average-price baseline; closer to 1 is better, while a negative value would be worse than that baseline.


In [ ]:
rows = []
predictions = {}
for name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    predicted = pipeline.predict(X_test)
    predictions[name] = predicted
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predicted),
        "MSE": mean_squared_error(y_test, predicted),
        "RMSE": np.sqrt(mean_squared_error(y_test, predicted)),
        "R2": r2_score(y_test, predicted),
    })

comparison = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
display(comparison.style.format({"MAE": "${:,.0f}", "MSE": "{:,.0f}", "RMSE": "${:,.0f}", "R2": "{:.3f}"}))
best_name = comparison.loc[0, "Model"]
print("Best model by held-out RMSE:", best_name)


### Visualize errors and prediction fit
We use RMSE to choose a model because it is in dollars and penalizes large misses. MAE is a useful companion measure of typical absolute error. No single metric tells the entire story.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
comparison.plot.bar(x="Model", y="RMSE", ax=axes[0], legend=False, color="#2878B5")
axes[0].set(title="Held-out RMSE (lower is better)", ylabel="RMSE ($)", xlabel="")
axes[0].tick_params(axis="x", rotation=35)
best_predictions = predictions[best_name]
axes[1].scatter(y_test, best_predictions, alpha=0.55, color="#66A182")
limits = [min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())]
axes[1].plot(limits, limits, "--", color="#D14A3B")
axes[1].set(title=f"Actual vs predicted: {best_name}", xlabel="Actual price ($)", ylabel="Predicted price ($)")
plt.tight_layout()
plt.show()


### Interpretation
The table ranks models on exactly the same held-out sales. The lowest RMSE is our simple selection rule; compare MAE and R² as supporting context. A more flexible model can capture nonlinear patterns, but its score does not guarantee accuracy for every home or future market. Notebook 4 uses this same best estimator to inspect errors and then refits the selected pipeline on all labeled rows for the saved app.


# 4. Final Model and Predictions

## Select, inspect, and save
We select the model with the lowest RMSE from Notebook 3's held-out comparison. First, we inspect that model's held-out predictions and residuals. Then we refit a fresh copy of the chosen pipeline on all 1,460 labeled sales and save the complete pipeline. The held-out metrics remain the honest performance estimate; refitting on all labeled rows gives the app as much training data as available.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "train.csv").exists():
    ROOT = Path.cwd().parent
TRAIN_PATH = ROOT / "data" / "raw" / "train.csv"
TEST_PATH = ROOT / "data" / "raw" / "test.csv"
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(f"Project folder: {ROOT}")
print(f"Training rows: {len(train):,} | competition test rows: {len(test):,}")


In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


### Recreate the same comparison and holdout
We repeat the model definitions and fixed split so this notebook runs on its own after Notebook 3. Reusing a fixed random seed makes the model selection reproducible.


In [ ]:
X = train[FEATURES]
y = train[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Lasso Regression": Lasso(alpha=100.0, max_iter=20000),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=1),
}
comparison_rows = []
fitted = {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", build_preprocessor()), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    comparison_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mean_squared_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    })
comparison = pd.DataFrame(comparison_rows).sort_values("RMSE").reset_index(drop=True)
best_name = comparison.loc[0, "Model"]
best_holdout_model = fitted[best_name]
best_prediction = best_holdout_model.predict(X_test)
holdout_metrics = comparison.loc[0, ["MAE", "MSE", "RMSE", "R2"]].to_dict()
print("Selected:", best_name)
display(comparison)


### Residual analysis
A residual is actual price minus predicted price. Residuals centered around zero suggest there is no broad over- or under-prediction. A pattern or widening spread can reveal errors that vary by price or property type. This is a diagnostic, not a guarantee that every segment is equally accurate.


In [ ]:
residuals = y_test.to_numpy() - best_prediction
holdout_output = pd.DataFrame({"Actual": y_test.to_numpy(), "Predicted": best_prediction, "Residual": residuals})
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(best_prediction, residuals, alpha=0.55, color="#2878B5")
axes[0].axhline(0, color="#D14A3B", linestyle="--")
axes[0].set(title="Residuals by predicted price", xlabel="Predicted price ($)", ylabel="Actual − predicted ($)")
axes[1].hist(residuals, bins=25, color="#E6A857", edgecolor="white")
axes[1].axvline(0, color="#D14A3B", linestyle="--")
axes[1].set(title="Distribution of residuals", xlabel="Residual ($)", ylabel="Number of homes")
plt.tight_layout()
plt.show()
print("Mean residual:", f"${residuals.mean():,.0f}")
print("MAE:", f"${holdout_metrics['MAE']:,.0f}", "| RMSE:", f"${holdout_metrics['RMSE']:,.0f}", "| R²:", f"{holdout_metrics['R2']:.3f}")


### Refit on all labeled homes and save
The saved joblib object includes both the preprocessing and estimator. The competition test file remains unseen and is not used for model selection. We save the held-out comparison, metrics, and predictions so the app can show exactly how the model performed.


In [ ]:
model_dir = ROOT / "models"
processed_dir = ROOT / "data" / "processed"
model_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

final_pipeline = clone(best_holdout_model)
final_pipeline.fit(X, y)
joblib.dump(final_pipeline, model_dir / "ames_price_pipeline.joblib")
comparison.to_csv(processed_dir / "model_comparison.csv", index=False)
holdout_output.to_csv(processed_dir / "holdout_predictions.csv", index=False)
metadata = {
    "best_model": best_name,
    "selection_metric": "RMSE",
    "holdout_metrics": {key: float(value) for key, value in holdout_metrics.items()},
    "train_rows": int(len(X)),
    "holdout_rows": int(len(X_test)),
    "features": FEATURES,
}
(model_dir / "model_metrics.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved complete pipeline:", model_dir / "ames_price_pipeline.joblib")
print("Saved evaluation files:", processed_dir)


### Strengths and limitations
The model has a clear held-out check, uses a small set of meaningful inputs, and packages its learned preprocessing with the estimator. It cannot know unprovided details or future market changes, and the held-out set is modest. The estimate is a learning demonstration, not a professional appraisal.


# 5. Model Interpretation

## What feature importance can and cannot tell us
Some estimators expose a ranking of features they use. For a tree model, importance summarizes how much splits on each transformed input helped reduce error. For a linear model, coefficient size and direction describe the fitted relationship after scaling, but correlated inputs can make individual coefficients unstable.

Importance is not causation, and one-hot encoding creates multiple columns from one original category. We will summarize each category's encoded columns back to its original input name for a readable chart.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "train.csv").exists():
    ROOT = Path.cwd().parent
TRAIN_PATH = ROOT / "data" / "raw" / "train.csv"
TEST_PATH = ROOT / "data" / "raw" / "test.csv"
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(f"Project folder: {ROOT}")
print(f"Training rows: {len(train):,} | competition test rows: {len(test):,}")


In [ ]:
import joblib
import matplotlib.pyplot as plt

model_path = ROOT / "models" / "ames_price_pipeline.joblib"
pipeline = joblib.load(model_path)
preprocessor = pipeline.named_steps["preprocess"]
estimator = pipeline.named_steps["model"]
feature_names = preprocessor.get_feature_names_out()
print("Loaded model:", type(estimator).__name__)
print("Transformed input columns:", len(feature_names))


### Calculate a simple importance score
For a tree estimator, we use its built-in impurity-based importance. For a linear estimator, we use the absolute value of its standardized coefficient as a rough strength score. The sign of a linear coefficient indicates the fitted direction, but not a causal effect.


In [ ]:
if hasattr(estimator, "feature_importances_"):
    raw_importance = estimator.feature_importances_
    importance_method = "Tree split importance"
elif hasattr(estimator, "coef_"):
    raw_importance = np.abs(np.ravel(estimator.coef_))
    importance_method = "Absolute standardized coefficient"
else:
    raise TypeError("Selected estimator does not expose a simple importance measure.")

def original_feature(transformed_name):
    for feature in FEATURES:
        if transformed_name == feature or transformed_name.startswith(feature + "_"):
            return feature
    return transformed_name

importance = pd.DataFrame({"TransformedFeature": feature_names, "Importance": raw_importance})
importance["Feature"] = importance["TransformedFeature"].map(original_feature)
importance = importance.groupby("Feature", as_index=False)["Importance"].sum().sort_values("Importance", ascending=False)
importance["Importance"] = importance["Importance"] / importance["Importance"].sum()
print("Ranking method:", importance_method)
display(importance.head(15).style.format({"Importance": "{:.1%}"}))


### Visualize the strongest inputs
This chart combines the encoded columns belonging to the same original category, such as all neighborhood indicators. The ranking is specific to this sample, selected model, and chosen input set.


In [ ]:
top = importance.head(12).sort_values("Importance")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["Feature"], top["Importance"], color="#2878B5")
ax.set(title=f"Most influential inputs ({importance_method.lower()})", xlabel="Share of summed importance")
plt.tight_layout()
plt.show()
output_path = ROOT / "data" / "processed" / "feature_importance.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
importance.to_csv(output_path, index=False)
print("Saved app feature chart data:", output_path)


### Takeaways
Quality, living area, location, and age are plausible strong signals in this housing dataset; the computed ranking above shows how the selected model used them. If two inputs move together, importance may be shared between them. Use the model comparison and residual plots alongside this chart rather than treating importance as a complete explanation of an individual price.
